In [2]:
import pandas as pd
import numpy as np
import requests
from urllib.parse import urlparse

# 1. Load the PhiUSIIL Dataset
data_path = "../data/PhiUSIIL_Phishing_URL_Dataset.csv"
df_full = pd.read_csv(data_path)

# 2. Take a 50,000-row random sample (random_state=42 guarantees the exact same 50k rows every time)
df = df_full.sample(n=50000, random_state=42).reset_index(drop=True)

print(f"Total Rows Sampled: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
display(df.head(3))

Total Rows Sampled: 50000
Total Columns: 56


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,827598.txt,https://www.northcm.ac.th,24,www.northcm.ac.th,17,0,th,100.000000,0.800000,0.000510,...,0,0,1,47,3,4,69,7,115,1
1,oph00308.txt,https://unitedmartialartscenters.com/at0/mygov...,59,unitedmartialartscenters.com,28,0,com,33.522664,1.000000,0.522907,...,1,0,1,0,2,0,0,0,1,0
2,8112660.txt,https://email.mail1.onesignal.os.tc/c/ejwcz02u...,385,email.mail1.onesignal.os.tc,27,0,tc,3.199329,0.458333,0.000046,...,0,0,0,0,0,0,0,0,1,0


In [3]:
# Known URL shortening domains
SHORTENER_DOMAINS = {
    'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 
    'is.gd', 'buff.ly', 'cutt.ly', 'rb.gy', 'shorturl.at'
}

def resolve_short_url(url, timeout=5):
    """
    Phase 1 Module: Unrolls shortened URLs via HTTP HEAD/GET redirect headers
    and extracts the true destination URL, root domain, and redirect hop count.
    """
    try:
        parsed_initial = urlparse(url)
        initial_domain = parsed_initial.netloc.lower().replace("www.", "")
        is_shortened = initial_domain in SHORTENER_DOMAINS
        
        # Trace HTTP redirects without downloading heavy page content
        response = requests.head(url, allow_redirects=True, timeout=timeout)
        final_url = response.url
        redirect_hops = len(response.history)
        final_domain = urlparse(final_url).netloc.lower().replace("www.", "")
        
        return {
            "Initial_URL": url,
            "Is_Shortener_Service": is_shortened,
            "Resolved_Final_URL": final_url,
            "Extracted_Root_Domain": final_domain,
            "NoOfURLRedirect": redirect_hops
        }
    except Exception as e:
        return {
            "Initial_URL": url,
            "Error": str(e),
            "Resolved_Final_URL": url,
            "NoOfURLRedirect": 0
        }

# Quick verification test (Safe offline/online demo for your guide)
print("Short-URL Redirect Resolution Module initialized successfully.")

Short-URL Redirect Resolution Module initialized successfully.


In [4]:
# 1. Clean the Domain column (lowercase, strip 'www.') to serve as our strict Grouping ID
groups = df['Domain'].astype(str).str.lower().str.replace(r'^www\.', '', regex=True)

# 2. Extract the Target Label (y) -> 1 = Legitimate, 0 = Phishing (in PhiUSIIL)
y = df['label']

# 3. Drop non-numeric/raw text columns and the target label to form our pure numerical Feature Matrix (X)
text_and_target_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']
cols_to_drop = [col for col in text_and_target_cols if col in df.columns]

X = df.drop(columns=cols_to_drop)

# 4. Verify Phase 1 Output
print("--- PHASE 1 COMPLETE ---")
print(f"Feature Matrix (X) shape : {X.shape} (Pure numerical features)")
print(f"Target Vector (y) shape  : {y.shape}")
print(f"Unique Root Domains      : {groups.nunique()} out of {len(df)} total URLs")
print(f"\nNotice how {len(df) - groups.nunique()} URLs share duplicate root domains! This proves why random splitting causes leakage.")

--- PHASE 1 COMPLETE ---
Feature Matrix (X) shape : (50000, 50) (Pure numerical features)
Target Vector (y) shape  : (50000,)
Unique Root Domains      : 47790 out of 50000 total URLs

Notice how 2210 URLs share duplicate root domains! This proves why random splitting causes leakage.


In [5]:

from sklearn.model_selection import train_test_split, GroupShuffleSplit


# TRACK 2A: The Flawed Baseline (Random Split)
# This blindly shuffles the data. Domains will leak across train and test sets.
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# TRACK 2B: The Leakage-Aware Audit (Domain-Grouped Split)
# This ensures 100% of a domain goes entirely to Train OR entirely to Test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# gss.split gives us the row index numbers for train and test sets
for train_idx, test_idx in gss.split(X, y, groups):
    X_train_B = X.iloc[train_idx]
    X_test_B = X.iloc[test_idx]
    y_train_B = y.iloc[train_idx]
    y_test_B = y.iloc[test_idx]
    
    # We also slice the domains just so we can verify the split worked perfectly
    domains_train_B = groups.iloc[train_idx]
    domains_test_B = groups.iloc[test_idx]

# VERIFICATION (Proving it to the Reviewers)
print("--- PHASE 2 COMPLETE ---")
print(f"Track 2A (Random) Train Size: {len(X_train_A)} | Test Size: {len(X_test_A)}")
print(f"Track 2B (Grouped) Train Size: {len(X_train_B)} | Test Size: {len(X_test_B)}\n")

# Mathematical proof of leakage vs zero-leakage
overlap_A = set(groups.loc[X_train_A.index]).intersection(set(groups.loc[X_test_A.index]))
overlap_B = set(domains_train_B).intersection(set(domains_test_B))

print(f"Track 2A Domain Leakage: {len(overlap_A)} root domains leaked into the test set!")
print(f"Track 2B Domain Leakage: {len(overlap_B)} root domains leaked. (Perfect zero-day environment!)")

--- PHASE 2 COMPLETE ---
Track 2A (Random) Train Size: 40000 | Test Size: 10000
Track 2B (Grouped) Train Size: 40191 | Test Size: 9809

Track 2A Domain Leakage: 246 root domains leaked into the test set!
Track 2B Domain Leakage: 0 root domains leaked. (Perfect zero-day environment!)
